# AnnData / single-cell API demonstration

Most public scRNA-seq datasets do **not** contain defensible Gaussian per-cell/per-gene error bars. This offline, executable notebook therefore creates a small count-like AnnData object and an explicitly **illustrative** uncertainty layer. The error bars below are not experimentally inferred and the output must not be interpreted as a biological result.

In [ ]:
import anndata as ad
import numpy as np

import noisecov as nc

rng = np.random.default_rng(12)
counts = rng.negative_binomial(4, 0.45, size=(250, 60))
log_expression = np.log1p(counts).astype(float)
adata = ad.AnnData(X=counts)
adata.var_names = [f"gene_{i:03d}" for i in range(adata.n_vars)]
adata.layers["measurement_mean"] = log_expression
# API demonstration only: this is not an inferred experimental uncertainty.
illustrative_sd = 0.15 + 0.65 / np.sqrt(counts + 1.0)
illustrative_sd *= rng.lognormal(0, 0.15, size=counts.shape)
adata.layers["measurement_sd"] = illustrative_sd
adata.var["highly_variable"] = np.var(log_expression, axis=0).argsort().argsort() >= 40
adata

In [ ]:
selected_genes = adata.var_names[adata.var["highly_variable"]]
result = nc.from_anndata(
    adata,
    x_layer="measurement_mean",
    sd_layer="measurement_sd",
    genes=selected_genes,
    method="corrected",
    write=True,
)
result.covariance.shape, result.adata.varp.keys(), "latent_covariance" in adata.varp

The last value is false: writing returned a subset copy and did not overwrite the input. For a real analysis, replace `measurement_sd` only with calibrated uncertainty on the same scale as `measurement_mean`. Raw UMI counts require count-based modeling.

## Optional scvi-tools bridge

An upstream scvi-tools model can produce posterior draws of a precisely defined expression quantity. Their mean and standard deviation across draws can be an approximate input only when a Gaussian summary is scientifically defensible. Posterior uncertainty, posterior-predictive count variability, and measurement error are not interchangeable; posterior cross-gene dependence is also lost when only marginal SDs are retained.

In [ ]:
# Stand-in for samples returned by an upstream probabilistic model.
# Replace this tensor only with draws of a clearly defined expression quantity.
illustrative_posterior_draws = rng.normal(
    loc=log_expression[None, :, :],
    scale=illustrative_sd[None, :, :],
    size=(50, *log_expression.shape),
)
posterior_mean, posterior_sd = nc.posterior_samples_to_gaussian(illustrative_posterior_draws)
posterior_mean.shape, posterior_sd.shape